<a id="nlp-overview"></a>

$$
\text{非线性规划 (Nonlinear Programming, NLP): 目标函数或约束中至少有一个是非线性的} \\
\min_x f(x) \quad \text{s.t.} \quad g_i(x) \leq 0 \ (i = 1, \cdots, m), \quad h_j(x) = 0 \ (j = 1, \cdots, p) \\ \quad \\
\text{和线性规划的区别:} \\
\text{1. 最优解不一定在顶点上, 可以在可行域内部, 也可以在边界的任意位置} \\
\text{2. 可能有多个局部最优解 (Local Optimum), 算法通常只能保证找到局部最优} \\
\text{3. 凸问题 (Convex Problem) 例外: 局部最优 = 全局最优 (下面证明)} \\ \quad \\
\text{分三种情况: 无约束, 等式约束 (拉格朗日乘子法), 不等式约束 (KKT 条件)}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from scipy.optimize import minimize

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

<a id="unconstrained"></a>

$$
\text{1. 无约束优化: 最优点满足 } \nabla f(x^*) = 0 \text{ (一阶必要条件), Hessian 正定 (二阶充分条件)} \\ \quad \\
\text{例: Rosenbrock 函数 (香蕉函数), 最小值在 (1, 1), 函数值 0} \\
f(x, y) = (1 - x)^2 + 100 (y - x^2)^2 \\
\nabla f = \begin{bmatrix} -2(1 - x) - 400 x (y - x^2) \\ 200 (y - x^2) \end{bmatrix}, \quad
H = \begin{bmatrix} 2 - 400 y + 1200 x^2 & -400 x \\ -400 x & 200 \end{bmatrix} \\
\text{在 (1, 1) 处 } H = \begin{bmatrix} 802 & -400 \\ -400 & 200 \end{bmatrix} \text{, 条件数很大 (山谷又窄又弯)} \\ \quad \\
\text{梯度下降: } x_{k+1} = x_k - \eta \nabla f \text{, 学习率要小于 } 2 / \lambda_{\max} \text{, 在窄谷里来回震荡, 很慢} \\
\text{牛顿法: } x_{k+1} = x_k - H^{-1} \nabla f \text{, 用曲率信息把窄谷 "拉圆", 收敛快很多}
$$

数学基础: [梯度下降与学习率上限](../math_basics/calculus.ipynb#gradient-descent) · [牛顿法](../math_basics/calculus.ipynb#newton) · [极值的二阶条件](../math_basics/calculus.ipynb#extremum)

In [ ]:
def rosen(p):
    x, y = p
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosen_grad(p):
    x, y = p
    return np.array([-2 * (1 - x) - 400 * x * (y - x**2), 200 * (y - x**2)])

def rosen_hess(p):
    x, y = p
    return np.array([[2 - 400 * y + 1200 * x**2, -400 * x], [-400 * x, 200]])

H_star = rosen_hess([1, 1])
eig = np.linalg.eigvalsh(H_star)
print(f'(1, 1) 处 Hessian 特征值 = {eig}, 条件数 = {eig[-1] / eig[0]:.1f}, 梯度下降学习率上限 2/lambda_max = {2 / eig[-1]:.5f}')

start = np.array([-1.2, 1.0])

# 梯度下降
p, path_gd = start.copy(), [start.copy()]
for k in range(20000):
    p = p - 0.0015 * rosen_grad(p)
    path_gd.append(p.copy())
    if np.linalg.norm(rosen_grad(p)) < 1e-6:
        break
path_gd = np.array(path_gd)
print(f'梯度下降: {len(path_gd) - 1} 步 (上限 20000 步, 还没收敛到梯度 < 1e-6), 终点 = {np.round(path_gd[-1], 6)}, f = {rosen(path_gd[-1]):.2e}')

# 牛顿法
p, path_nt = start.copy(), [start.copy()]
for k in range(100):
    p = p - np.linalg.solve(rosen_hess(p), rosen_grad(p))
    path_nt.append(p.copy())
    if np.linalg.norm(rosen_grad(p)) < 1e-10:
        break
path_nt = np.array(path_nt)
print(f'牛顿法:   {len(path_nt) - 1} 步, 终点 = {np.round(path_nt[-1], 6)}, f = {rosen(path_nt[-1]):.2e}')

res_bfgs = minimize(rosen, start, jac=rosen_grad, method='BFGS')
print(f'scipy BFGS: {res_bfgs.nit} 步, 终点 = {np.round(res_bfgs.x, 6)}')

xs, ys = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
plt.figure(figsize=(4, 3))
plt.contour(xs, ys, rosen([xs, ys]), levels=np.logspace(-1, 3, 15), alpha=0.5)
plt.plot(path_gd[::200, 0], path_gd[::200, 1], 'b.-', markersize=3, label='gradient descent')
plt.plot(path_nt[:, 0], path_nt[:, 1], 'ro-', markersize=4, label='Newton')
plt.plot(1, 1, 'k*', markersize=12, label='minimum (1, 1)')
plot_nicely()
plt.show()

<a id="convexity"></a>

$$
\text{凸函数 (Convex Function): 任意两点连线上, 函数值不超过连线} \\
f(\lambda x + (1 - \lambda) y) \leq \lambda f(x) + (1 - \lambda) f(y), \quad 0 \leq \lambda \leq 1 \\ \quad \\
\text{1. 一阶条件: 切线 (切平面) 在函数下方} \\
\text{由定义: } f(x + \lambda (y - x)) \leq f(x) + \lambda (f(y) - f(x)) \implies \frac{f(x + \lambda(y - x)) - f(x)}{\lambda} \leq f(y) - f(x) \\
\lambda \to 0 \text{, 左边是方向导数 } \nabla f(x)^T (y - x) \implies f(y) \geq f(x) + \nabla f(x)^T (y - x) \\
\text{推论: 若 } \nabla f(x^*) = 0 \text{, 则对所有 y, } f(y) \geq f(x^*) \text{, 驻点就是全局最小值} \\ \quad \\
\text{2. 二阶条件: Hessian 处处半正定} \iff \text{凸函数} \\
\text{沿任意直线 } g(t) = f(x + t d) \text{, } g''(t) = d^T H d \geq 0 \text{, 一元函数二阶导非负就是向上弯} \\ \quad \\
\text{3. 凸优化问题: f 是凸函数, 可行域是凸集 (例如 } g_i \text{ 是凸函数, } h_j \text{ 是线性函数)} \\
\text{定理: 凸优化问题的局部最优解就是全局最优解} \\
\text{反证: 设 } x^* \text{ 是局部最优, 但存在可行的 y 使 } f(y) < f(x^*) \\
\text{连线上的点 } z = \lambda y + (1 - \lambda) x^* \text{ 可行 (可行域是凸集), 且 } f(z) \leq \lambda f(y) + (1 - \lambda) f(x^*) < f(x^*) \\
\lambda \to 0 \text{ 时 z 可以任意接近 } x^* \text{, 与 } x^* \text{ 是局部最优矛盾} \\
\text{(线性规划是凸优化的特例, 所以单纯形法找到的局部最优就是全局最优)}
$$

数学基础: [方向导数](../math_basics/calculus.ipynb#gradient) · [正定矩阵](../math_basics/linear_algebra.ipynb#positive-definite) · [多元泰勒展开](../math_basics/calculus.ipynb#multivariate-taylor)

In [ ]:
def convex_f(p):
    x, y = p
    return x**2 + x * y + 2 * y**2 + np.exp(0.5 * x) - 3 * y

def nonconvex_f(p):
    x, y = p
    return np.sin(3 * x) + np.cos(3 * y) + 0.1 * (x**2 + y**2)

def numerical_hessian(f, p, h=1e-4):
    p = np.asarray(p, float); n = len(p); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei, ej = np.eye(n)[i] * h, np.eye(n)[j] * h
            H[i, j] = (f(p + ei + ej) - f(p + ei - ej) - f(p - ei + ej) + f(p - ei - ej)) / (4 * h * h)
    return H

rng = np.random.default_rng(0)
pts = rng.uniform(-3, 3, size=(300, 2))
min_eig_convex = min(np.linalg.eigvalsh(numerical_hessian(convex_f, q))[0] for q in pts)
min_eig_nonconvex = min(np.linalg.eigvalsh(numerical_hessian(nonconvex_f, q))[0] for q in pts)
print(f'凸函数: 300 个随机点 Hessian 最小特征值 = {min_eig_convex:.4f} (>= 0)')
print(f'非凸函数: 300 个随机点 Hessian 最小特征值 = {min_eig_nonconvex:.4f} (有负的)')

# 从 30 个随机起点出发做局部优化
starts = rng.uniform(-3, 3, size=(30, 2))
convex_ends = np.array([minimize(convex_f, s).x for s in starts])
nonconvex_vals = np.array([minimize(nonconvex_f, s).fun for s in starts])
print(f'凸函数: 30 个起点的终点散布 (最大距离) = {np.max(np.linalg.norm(convex_ends - convex_ends[0], axis=1)):.2e}, 都到同一个全局最优')
print(f'非凸函数: 30 个起点得到 {len(np.unique(np.round(nonconvex_vals, 4)))} 个不同的局部最优值, 最小 = {nonconvex_vals.min():.4f}, 最大 = {nonconvex_vals.max():.4f}')

# 一阶条件: 切平面在凸函数下方
x0 = np.array([0.5, -0.3])
g0 = np.array([(convex_f(x0 + e) - convex_f(x0 - e)) / 2e-6 for e in np.eye(2) * 1e-6])
ys = rng.uniform(-3, 3, size=(5000, 2))
gap = np.array([convex_f(y) for y in ys]) - (convex_f(x0) + (ys - x0) @ g0)
print(f'f(y) - [f(x) + grad^T (y - x)] 的最小值 = {gap.min():.4f} (>= 0)')

<a id="kkt"></a>

$$
\text{2. 不等式约束: KKT 条件 (Karush-Kuhn-Tucker)} \\
\min f(x) \quad \text{s.t.} \quad g(x) \leq 0 \\ \quad \\
\text{在最优点 } x^* \text{ 只有两种可能:} \\
\text{(a) 约束不起作用 (inactive): } g(x^*) < 0 \text{, } x^* \text{ 在可行域内部, 周围都可行, 和无约束一样 } \nabla f(x^*) = 0 \\
\text{(b) 约束起作用 (active): } g(x^*) = 0 \text{, 在边界上, 和等式约束一样 (拉格朗日) } \nabla f = -\mu \nabla g \\
\text{并且 } \mu \text{ 的符号是确定的: } \nabla g \text{ 指向 g 增大的方向 = 可行域外面, 往里走的方向是 } -\nabla g \\
\text{往可行域里面走不能让 f 下降 (否则 } x^* \text{ 不是最优): } \nabla f^T (-\nabla g) \geq 0 \\
\text{代入 } \nabla f = -\mu \nabla g \text{: } \mu |\nabla g|^2 \geq 0 \implies \mu \geq 0 \\
\text{几何意义: } -\nabla f \text{ (f 下降最快的方向) 指向可行域外面, 想出去但被边界挡住了} \\ \quad \\
\text{两种情况统一写成 (情况 a 取 } \mu = 0 \text{):} \\
\begin{cases}
\nabla f(x^*) + \mu \nabla g(x^*) = 0 & \text{(稳定性 Stationarity)} \\
g(x^*) \leq 0 & \text{(原始可行 Primal Feasibility)} \\
\mu \geq 0 & \text{(对偶可行 Dual Feasibility)} \\
\mu \, g(x^*) = 0 & \text{(互补松弛 Complementary Slackness: 要么 } \mu = 0 \text{, 要么 } g = 0 \text{)}
\end{cases} \\ \quad \\
\text{多个约束: } \nabla f + \sum_i \mu_i \nabla g_i + \sum_j \lambda_j \nabla h_j = 0, \ \mu_i \geq 0, \ \mu_i g_i = 0 \text{ (} \lambda_j \text{ 没有符号要求)} \\
\text{和线性规划的对偶比较: 互补松弛是同一个条件, } \mu_i \text{ 就是影子价格} \\
\text{对凸问题, KKT 条件是充分必要的; 一般问题只是必要条件 (还需要约束规范性 constraint qualification)}
$$

数学基础: [拉格朗日乘子法](../math_basics/calculus.ipynb#lagrange) · [梯度与等高线](../math_basics/calculus.ipynb#gradient)

<a id="kkt-example"></a>

$$
\text{例: } \min f = (x - 3)^2 + (y - 2)^2 \quad \text{s.t.} \quad g_1 = x + y - 3 \leq 0, \quad g_2 = x - 1.5 \leq 0 \\
\nabla f = (2(x - 3), \ 2(y - 2)), \quad \nabla g_1 = (1, 1), \quad \nabla g_2 = (1, 0) \\
\text{两个约束, 每个 active 或 inactive, 共 } 2^2 = 4 \text{ 种情况, 逐个检查:} \\ \quad \\
\text{1. 都不起作用 } (\mu_1 = \mu_2 = 0) \text{: } \nabla f = 0 \implies (3, 2) \text{, 但 } g_1 = 2 > 0 \text{, 不可行} \\
\text{2. 只有 } g_1 \text{ 起作用 } (\mu_2 = 0) \text{: } 2(x - 3) + \mu_1 = 0, \ 2(y - 2) + \mu_1 = 0 \implies x - 3 = y - 2 \\
\quad \text{代入 } x + y = 3 \implies (2, 1), \ \mu_1 = 2 \text{, 但 } g_2 = 0.5 > 0 \text{, 不可行} \\
\text{3. 只有 } g_2 \text{ 起作用 } (\mu_1 = 0) \text{: } x = 1.5, \ 2(1.5 - 3) + \mu_2 = 0 \implies \mu_2 = 3, \ y = 2 \text{, 但 } g_1 = 0.5 > 0 \text{, 不可行} \\
\text{4. 都起作用: } x = 1.5, \ y = 1.5 \\
\quad 2(1.5 - 3) + \mu_1 + \mu_2 = 0, \quad 2(1.5 - 2) + \mu_1 = 0 \implies \mu_1 = 1, \ \mu_2 = 2 \text{, 都} \geq 0 \text{, 满足 KKT} \\ \quad \\
\text{最优解 } (1.5, 1.5), \ f^* = 2.25 + 0.25 = 2.5 \\
\text{敏感性: 约束 } g_i \leq b_i \text{ 放宽一点, } \frac{\partial f^*}{\partial b_i} = -\mu_i \text{ (目标是最小化, 放宽约束让 f 减小)}
$$

In [ ]:
# f 是二次函数, 约束是线性的, 每种情况的 KKT 方程都是线性方程组
G_lin = np.array([[1.0, 1.0], [1.0, 0.0]])
b_lin = np.array([3.0, 1.5])
target = np.array([3.0, 2.0])

def solve_kkt(b_vec, verbose=True):
    best = None
    for active in product([False, True], repeat=2):
        idx = [i for i in range(2) if active[i]]
        k = len(idx)
        # 未知数 [x, y, mu_active...]: 2(x - target) + G_act^T mu = 0, G_act x = b_act
        M = np.zeros((2 + k, 2 + k))
        rhs = np.zeros(2 + k)
        M[:2, :2] = 2 * np.eye(2)
        rhs[:2] = 2 * target
        if k:
            M[:2, 2:] = G_lin[idx].T
            M[2:, :2] = G_lin[idx]
            rhs[2:] = b_vec[idx]
        sol = np.linalg.solve(M, rhs)
        xy, mu = sol[:2], np.zeros(2)
        mu[idx] = sol[2:]
        feasible = np.all(G_lin @ xy <= b_vec + 1e-9)
        dual_ok = np.all(mu >= -1e-9)
        f_val = np.sum((xy - target)**2)
        if verbose:
            print(f'active = {[f"g{i + 1}" for i in idx]}: (x, y) = {np.round(xy, 4)}, mu = {np.round(mu, 4)}, '
                  f'可行? {feasible}, mu >= 0? {dual_ok}, f = {f_val:.4f}')
        if feasible and dual_ok:
            best = (xy, mu, f_val)
    return best

xy_kkt, mu_kkt, f_kkt = solve_kkt(b_lin)
print(f'\nKKT 解: (x, y) = {xy_kkt}, mu = {mu_kkt}, f* = {f_kkt:.4f}')

cons = [{'type': 'ineq', 'fun': lambda p, i=i: b_lin[i] - G_lin[i] @ p} for i in range(2)]   # scipy 的 ineq 约定是 fun >= 0
res_slsqp = minimize(lambda p: np.sum((p - target)**2), x0=[0, 0], method='SLSQP', constraints=cons)
print(f'scipy SLSQP: (x, y) = {np.round(res_slsqp.x, 6)}, f* = {res_slsqp.fun:.6f}, 一致? {np.allclose(res_slsqp.x, xy_kkt, atol=1e-6)}')

# 敏感性 df*/db_i = -mu_i
for i in range(2):
    db = np.zeros(2); db[i] = 1e-4
    f_plus = solve_kkt(b_lin + db, verbose=False)[2]
    print(f'放宽 g{i + 1}: df*/db = {(f_plus - f_kkt) / 1e-4:.4f}, -mu = {-mu_kkt[i]:.4f}')

In [ ]:
# 非线性约束的几何意义: min x + y, s.t. x^2 + y^2 <= 1
# KKT: (1, 1) + mu (2x, 2y) = 0  =>  x = y = -1 / (2 mu), 代入 x^2 + y^2 = 1  =>  mu = 1/sqrt(2), x = y = -1/sqrt(2)
mu_c = 1 / np.sqrt(2)
x_c = np.array([-1, -1]) / np.sqrt(2)
res_c = minimize(lambda p: p[0] + p[1], x0=[0.5, 0], method='SLSQP',
                 constraints={'type': 'ineq', 'fun': lambda p: 1 - p[0]**2 - p[1]**2})
print(f'KKT 手算: x* = {np.round(x_c, 6)}, mu = {mu_c:.6f}; SLSQP: x* = {np.round(res_c.x, 6)}')
grad_f, grad_g = np.array([1.0, 1.0]), 2 * x_c
print(f'grad f + mu grad g = {np.round(grad_f + mu_c * grad_g, 10)}')

theta = np.linspace(0, 2 * np.pi, 200)
xs, ys = np.meshgrid(np.linspace(-1.8, 1.8, 200), np.linspace(-1.8, 1.8, 200))
plt.figure(figsize=(4, 3))
plt.fill(np.cos(theta), np.sin(theta), alpha=0.2, label='feasible x^2 + y^2 <= 1')
plt.contour(xs, ys, xs + ys, levels=10, alpha=0.5)
plt.quiver(*x_c, *(-grad_f), color='r', scale=6, label='-grad f (wants to go)')
plt.quiver(*x_c, *grad_g, color='b', scale=6, label='grad g (outward)')
plt.plot(*x_c, 'ko')
plt.gca().set_aspect('equal')
plot_nicely()
plt.show()

<a id="penalty"></a>

$$
\text{罚函数法 (Penalty Method): 把约束问题变成一串无约束问题} \\
\min_x \ P_\rho(x) = f(x) + \frac{\rho}{2} \sum_i \max(0, g_i(x))^2 \\
\text{违反约束时加上一个惩罚, 满足约束时不惩罚; } \rho \to \infty \text{ 时, 解趋近于原问题的解} \\ \quad \\
\text{和 KKT 的联系: } P_\rho \text{ 的最优点满足 } \nabla f + \sum_i \underbrace{\rho \max(0, g_i)}_{\approx \mu_i} \nabla g_i = 0 \\
\text{(用到 } \frac{d}{dg} \frac{1}{2}\max(0, g)^2 = \max(0, g) \text{)} \\
\text{和 KKT 的稳定性条件形式一样, 所以 } \rho \max(0, g_i(x_\rho)) \to \mu_i \text{, 罚函数法顺便估计了拉格朗日乘子} \\
\text{代价: 解 } x_\rho \text{ 一直稍微违反约束 } g_i \approx \mu_i / \rho \text{, 而且 } \rho \text{ 越大, 问题的条件数越大, 越难解}
$$

In [ ]:
def penalty_obj(p, rho):
    viol = np.maximum(0, G_lin @ p - b_lin)
    return np.sum((p - target)**2) + rho / 2 * np.sum(viol**2)

def penalty_grad(p, rho):
    viol = np.maximum(0, G_lin @ p - b_lin)
    return 2 * (p - target) + rho * G_lin.T @ viol

p = np.zeros(2)
for rho in [1, 10, 100, 1000, 10000, 100000]:
    p = minimize(penalty_obj, p, args=(rho,), jac=penalty_grad, method='BFGS', options={'gtol': 1e-10}).x   # 用上一次的解热启动
    viol = np.maximum(0, G_lin @ p - b_lin)
    print(f'rho = {rho:>6}: x = {np.round(p, 6)}, 违反量 = {np.round(viol, 6)}, mu 估计 = rho * 违反量 = {np.round(rho * viol, 4)}')
print(f'KKT 精确解: x = {xy_kkt}, mu = {mu_kkt}')

<a id="portfolio"></a>

$$
\text{应用: 投资组合优化 (Markowitz Portfolio)} \\
w_i \text{: 投资第 i 种资产的比例, } \mu \text{: 期望收益率向量, } \Sigma \text{: 收益率的协方差矩阵} \\
\text{组合收益 } R = w^T r \text{, 期望 } E[R] = w^T \mu \text{, 方差 } Var(R) = w^T \Sigma w \text{ (协方差矩阵那一节的公式)} \\ \quad \\
\min_w \ w^T \Sigma w \quad \text{s.t.} \quad w^T \mu \geq r_{target}, \quad \sum_i w_i = 1, \quad w_i \geq 0 \\
\text{目标是凸函数 (} \Sigma \text{ 半正定, Hessian = } 2\Sigma \text{), 约束是线性的, 所以是凸优化, SLSQP 找到的就是全局最优} \\
\text{改变 } r_{target} \text{, 每个目标收益对应一个最小风险, 连起来是有效前沿 (Efficient Frontier)}
$$

数学基础: [协方差矩阵与 Var(w^T x) = w^T Σ w](../math_basics/statistics.ipynb#covariance-matrix) · [矩阵求导](../math_basics/linear_algebra.ipynb#matrix-calculus)

In [ ]:
mu_r = np.array([0.08, 0.12, 0.15, 0.05])
vol = np.array([0.10, 0.18, 0.25, 0.04])
corr = np.array([[1.0, 0.3, 0.2, 0.0],
                 [0.3, 1.0, 0.5, 0.1],
                 [0.2, 0.5, 1.0, 0.0],
                 [0.0, 0.1, 0.0, 1.0]])
Sigma = corr * np.outer(vol, vol)

def solve_portfolio(r_target):
    cons = [{'type': 'eq', 'fun': lambda w: w.sum() - 1, 'jac': lambda w: np.ones_like(w)},
            {'type': 'ineq', 'fun': lambda w: w @ mu_r - r_target, 'jac': lambda w: mu_r}]
    return minimize(lambda w: w @ Sigma @ w, np.full(4, 0.25), jac=lambda w: 2 * Sigma @ w,
                    method='SLSQP', bounds=[(0, 1)] * 4, constraints=cons, options={'ftol': 1e-12})

res_p = solve_portfolio(0.10)
w = res_p.x
print(f'目标收益 10%: 权重 = {np.round(w, 4)}, 收益 = {w @ mu_r:.4f}, 风险 (标准差) = {np.sqrt(w @ Sigma @ w):.4f}')
w8 = solve_portfolio(0.08).x
print(f'同样要求 8% 的收益: 单独持有资产 1 的风险 = {vol[0]:.4f}, 最优组合的风险 = {np.sqrt(w8 @ Sigma @ w8):.4f} (分散投资降低风险)')

# 手动检查 KKT: 2 Sigma w = lam * 1 + nu * mu + eta, eta_i >= 0 只在 w_i = 0 处非零
active = w > 1e-6
A_kkt = np.column_stack([np.ones(active.sum()), mu_r[active]])
lam_nu, *_ = np.linalg.lstsq(A_kkt, 2 * Sigma[active] @ w, rcond=None)
print(f'KKT 稳定性残差 (非零权重的资产) = {np.max(np.abs(A_kkt @ lam_nu - 2 * Sigma[active] @ w)):.2e}, 收益约束的乘子 nu = {lam_nu[1]:.4f} (>= 0)')

targets = np.linspace(0.05, 0.15, 25)
risks = [np.sqrt(solve_portfolio(r).fun) for r in targets]
rng = np.random.default_rng(3)
W_rand = rng.dirichlet(np.ones(4), size=3000)
plt.figure(figsize=(4, 3))
plt.scatter(np.sqrt(np.einsum('ij,jk,ik->i', W_rand, Sigma, W_rand)), W_rand @ mu_r, s=2, alpha=0.3, label='random portfolios')
plt.plot(risks, targets, 'r-', label='efficient frontier')
plt.xlabel('risk (std)')
plt.ylabel('expected return')
plot_nicely()
plt.show()